# AI-Powered Inventory Management & Demand Forecasting
## Exploratory Data Analysis & Model Development Notebook

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.insert(0, '../backend')
from ml.preprocessing.data_cleaner import DataCleaner
from ml.feature_engineering.feature_builder import FeatureBuilder
from ml.training.model_trainer import ModelTrainer
from ml.evaluation.metrics import ModelEvaluator

## 1. Load Data

In [ ]:
df = pd.read_csv('../data/raw/sample_sales_data.csv', parse_dates=['date'])
print(f'Dataset shape: {df.shape}')
df.head()

## 2. Data Cleaning

In [ ]:
cleaner = DataCleaner(df)
df_clean = cleaner.clean()
print(f'Cleaned shape: {df_clean.shape}')

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
df_clean.groupby('date')['units_sold'].sum().plot(ax=axes[0,0], title='Daily Total Sales')
df_clean.groupby('category')['units_sold'].sum().plot(kind='bar', ax=axes[0,1], title='Sales by Category')
df_clean.groupby('product_name')['units_sold'].sum().nlargest(10).plot(kind='barh', ax=axes[1,0], title='Top 10 Products')
df_clean.groupby(df_clean['date'].dt.month)['units_sold'].sum().plot(kind='bar', ax=axes[1,1], title='Monthly Sales')
plt.tight_layout()
plt.show()

## 4. Feature Engineering

In [ ]:
builder = FeatureBuilder()
df_features = builder.build_features(df_clean)
print(f'Features shape: {df_features.shape}')
print(f'Features: {list(df_features.columns)}')

## 5. Model Training & Evaluation

In [ ]:
feature_cols = [c for c in df_features.columns if c not in ['units_sold', 'date', 'product_name', 'category', 'store_location']]
df_features = df_features.dropna()
split_idx = int(len(df_features) * 0.8)
train = df_features.iloc[:split_idx]
test = df_features.iloc[split_idx:]
X_train, y_train = train[feature_cols], train['units_sold']
X_test, y_test = test[feature_cols], test['units_sold']

In [ ]:
trainer = ModelTrainer()
evaluator = ModelEvaluator()
comparison = trainer.compare_models(X_train, y_train, X_test, y_test)
print('\nModel Comparison:')
for model_name, metrics in comparison.items():
    print(f'{model_name}: MAE={metrics["mae"]:.2f}, RMSE={metrics["rmse"]:.2f}, MAPE={metrics["mape"]:.2f}%')